# Corrigé · Peut-on reconnaître un manchot Adélie à partir de sa fiche de terrain ?

Une équipe de la station Palmer reçoit une fiche de l’île Dream. Le bec mesure 39,8 mm de long et 18,4 mm de haut, la nageoire mesure 190 mm et la fiche indique une femelle. La balance était en panne. **L’équipe doit-elle classer ce manchot comme Adélie ?**

Tu disposes de fiches d’archives dont l’espèce a été identifiée. **Entraîne une régression logistique** pour distinguer les Adélie des autres manchots. À la fin, donne à l’équipe ta prédiction pour cette fiche.

## 1. Que sait-on au moment de la prédiction ?

Charge les archives [penguins.csv](../../data/penguins.csv) avec la cellule ci-dessous. Repère la colonne qui contient la réponse connue. Relève les espèces présentes et le nombre de fiches de chaque espèce. Compte les valeurs manquantes par colonne.

Compare les ordres de grandeur de `bill_depth_mm` (hauteur du bec) et de `body_mass_g` (masse). Que signifierait un écart de 10 dans chacune de ces colonnes ?

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from enum import StrEnum

dossier_data = next(d for d in (Path("data"), Path("../data"), Path("../../data")) if d.is_dir())
manchots = pd.read_csv(dossier_data / "penguins.csv")

In [19]:
print("Fiches par espèce")
print(manchots["species"].value_counts().to_string())

print("\nValeurs manquantes par colonne")
print(manchots.isna().sum().to_string())

print("\nÉtendue des mesures")
print(manchots[["bill_depth_mm", "body_mass_g"]].agg(["min", "median", "max"]))


print("Un écart de 10g est faible pour la masse")
print("Un écart de 10mm est important pour un bec.")

Fiches par espèce
species
Adelie       152
Gentoo       124
Chinstrap     68

Valeurs manquantes par colonne
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0

Étendue des mesures
        bill_depth_mm  body_mass_g
min              13.1       2700.0
median           17.3       4050.0
max              21.5       6300.0
Un écart de 10g est faible pour la masse
Un écart de 10mm est important pour un bec.


L’archive contient 344 fiches avec une espèce connue. La hauteur du bec va de 13,1 à 21,5 mm, soit une étendue de 8,4 mm. Un écart de 10 mm dépasserait toute cette étendue. Pour la masse, un écart de 10 g est faible devant les 3 600 g qui séparent les extrêmes.

## 2. Comment vérifier que le modèle reconnaît de nouveaux manchots ?

Construis la cible `y` avec la valeur 1 pour `Adelie` et 0 pour les deux autres espèces. Pour `X`, garde les quatre mesures physiques, l’île et le sexe. Explique ce qui se passerait si tu laissais la colonne `species` dans les entrées.

Affiche l’effectif et la proportion d’Adélie dans chaque lot. Garde le test pour le bilan final.

In [ ]:
mesures = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
categories = ["island", "sex"]

X = manchots[mesures + categories]

y = manchots["species"].eq("Adelie").astype(int)

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_ap, X_val, y_ap, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42
)

for nom, cible in [
    ("Apprentissage", y_ap), 
    ("Validation", y_val), 
    ("Test", y_test)
]:
    print(f"{nom} : {cible.sum()} Adélie sur {len(cible)} fiches ({cible.mean():.1%})")

print("Inclure species dans X provoquerait un data leakage.")

Apprentissage : 91 Adélie sur 206 fiches (44.2%)
Validation : 31 Adélie sur 69 fiches (44.9%)
Test : 30 Adélie sur 69 fiches (43.5%)
Inclure species dans X provoquerait un data leakage.


La colonne `species` sert à construire la cible. La garder dans `X` donnerait au modèle la réponse pendant l’entraînement et l’évaluation, alors que l’équipe cherche justement à identifier une espèce inconnue. Le partage stratifié maintient la proportion d’Adélie autour de 44 % dans chacun des trois lots.

## 3. Que représente une masse standardisée ?

Sur la seule colonne `body_mass_g` du lot d’apprentissage, ajuste un `SimpleImputer` qui remplace les masses manquantes par la médiane. Sur les masses ainsi complétées, ajuste un `StandardScaler`. Relève la médiane mémorisée, la moyenne et l’écart-type utilisés par le scaler.

Choisis la première fiche de validation dont la masse est renseignée. Calcule sa masse standardisée à la main avec les paramètres appris, puis vérifie ton résultat avec `transform`. Que signifierait une valeur standardisée de −1 pour un manchot ?

In [21]:
imputation_masse = SimpleImputer(strategy="median") # instance d'objet de type SimpleImputer à laquelle on a passé en paramètre une valeur pour une propriété.

masses_ap = imputation_masse.fit_transform(X_ap[["body_mass_g"]])
scaler_masse = StandardScaler().fit(masses_ap)

mediane = imputation_masse.statistics_[0]
moyenne = scaler_masse.mean_[0]
ecart_type = scaler_masse.scale_[0]

print("Médiane :",mediane)
print("Moyenne :",moyenne)
print("Ecart-type :",ecart_type, end="\n\n")


fiche_val = X_val.loc[X_val["body_mass_g"].notna()].iloc[[0]]
masse = fiche_val["body_mass_g"].iloc[0]

z_manuel = (masse - moyenne) / ecart_type

masse_completee = imputation_masse.transform(fiche_val[["body_mass_g"]])
z_scaler = scaler_masse.transform(masse_completee)[0, 0]

print(f"Médiane : {mediane:.1f} g ; moyenne : {moyenne:.1f} g ; écart-type : {ecart_type:.1f} g")
print(f"Masse : {masse:.0f} g ; calcul à la main : {z_manuel:.4f} ; transform : {z_scaler:.4f}")

assert np.isclose(z_manuel, z_scaler)

print("Un Z de -1 pour le manchot, ça signifie un écart-type sous la moyenne.")

Médiane : 4050.0
Moyenne : 4208.859223300971
Ecart-type : 791.7833714889364

Médiane : 4050.0 g ; moyenne : 4208.9 g ; écart-type : 791.8 g
Masse : 5950 g ; calcul à la main : 2.1990 ; transform : 2.1990
Un Z de -1 pour le manchot, ça signifie un écart-type sous la moyenne.


La masse de 5 950 g se situe environ 2,20 écarts-types au-dessus de la moyenne d’apprentissage. Une valeur standardisée de −1 placerait le manchot un écart-type sous cette moyenne, soit environ 3 417 g.

## 4. Comment préparer une fiche entière ?

Construis un `ColumnTransformer` qui applique le traitement adapté à chaque colonne.

| Colonnes | Opérations à enchaîner |
|---|---|
| Longueur et hauteur du bec, longueur de la nageoire, masse | Compléter chaque mesure manquante avec sa médiane d’apprentissage, puis appliquer `StandardScaler`. |
| Île et sexe | Remplacer les valeurs manquantes par `Non renseigné`, puis utiliser `OneHotEncoder(handle_unknown="ignore")`. |

Ajuste cette préparation sur les entrées d’apprentissage. Transforme une fiche de validation, puis affiche les noms et les valeurs des colonnes obtenues. Retrouve la masse standardisée et la colonne qui indique l’île de cette fiche.

In [22]:
numerique = Pipeline([
    ("imputation", SimpleImputer(strategy="median")),
    ("standardisation", StandardScaler()),
])

categoriel = Pipeline([
    ("imputation", SimpleImputer(strategy="constant", fill_value="Non renseigné")),
    ("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preparation = ColumnTransformer([
    ("num", numerique, mesures),
    ("cat", categoriel, categories),
]).set_output(transform="pandas")

preparation.fit(X_ap)

fiche_transformee = preparation.transform(fiche_val)

print(fiche_transformee.T.to_string(header=False))

ile = fiche_val["island"].iloc[0]
print(f"\nÎle : {ile}, indiquée par cat__island_{ile} = 1.")

assert np.isclose(fiche_transformee["num__body_mass_g"].iloc[0], z_manuel)

num__bill_length_mm     1.037764
num__bill_depth_mm     -0.691442
num__flipper_length_mm  2.017990
num__body_mass_g        2.199012
cat__island_Biscoe      1.000000
cat__island_Dream       0.000000
cat__island_Torgersen   0.000000
cat__sex_Non renseigné  0.000000
cat__sex_female         0.000000
cat__sex_male           1.000000

Île : Biscoe, indiquée par cat__island_Biscoe = 1.


Pour cette fiche, `num__body_mass_g` vaut environ 2,199. On retrouve le calcul de l’étape 3. Parmi les colonnes de l’île, seule `cat__island_Biscoe` vaut 1.

## 5. La standardisation change-t-elle les prédictions ?

Construis deux pipelines qui acceptent les fiches brutes et se terminent par `LogisticRegression(max_iter=5000)`. Le premier utilise la préparation de l’étape 4. Pour le second, retire uniquement le scaler de la branche numérique. Garde l’imputation, l’encodage, le modèle et les lots identiques.

Entraîne les deux pipelines sur l’apprentissage. Pour chacun, compte les fiches de validation mal classées et calcule la proportion de prédictions correctes. Retiens celui qui fait le moins d’erreurs. En cas d’égalité, garde la version standardisée. Explique ton choix avec les résultats observés.

In [23]:
standardise = Pipeline([
    ("preparation", clone(preparation)),
    ("modele", LogisticRegression(max_iter=5000)),
])

sans_scaler = clone(standardise).set_params(
    preparation__num__standardisation="passthrough"
)

modeles = {"Standardisé": standardise, "Sans standardisation": sans_scaler}
erreurs_val = {}

for nom, modele in modeles.items():
    modele.fit(X_ap, y_ap)
    prediction = modele.predict(X_val)
    erreurs_val[nom] = (prediction != y_val).sum()
    exactitude = accuracy_score(y_val, prediction)
    print(f"{nom} : {erreurs_val[nom]} erreurs sur {len(y_val)}, exactitude {exactitude:.1%}")

if erreurs_val["Standardisé"] <= erreurs_val["Sans standardisation"]:
    choix = "Standardisé"
else:
    choix = "Sans standardisation"
retenu = modeles[choix]
print(f"Pipeline retenu : {choix}")

Standardisé : 0 erreurs sur 69, exactitude 100.0%
Sans standardisation : 0 erreurs sur 69, exactitude 100.0%
Pipeline retenu : Standardisé


Les deux pipelines classent correctement les 69 fiches de validation. On garde donc le pipeline standardisé, comme prévu en cas d’égalité.

## 6. Quelle réponse donner à l’équipe de terrain ?

Évalue une seule fois le pipeline retenu sur le test. Compte les Adélie manqués et les autres manchots classés à tort comme Adélie. Vérifie que la somme donne ton nombre total d’erreurs.

La fiche du début est reproduite ci-dessous. Transmets-la directement au pipeline retenu avec `predict`. Termine par deux phrases pour l’équipe. Donne la classe prédite pour ce manchot, puis situe la fiabilité de la décision à partir des erreurs observées sur le test.

In [24]:
prediction_test = retenu.predict(X_test)


matrice = confusion_matrix(
    y_test, prediction_test, labels=[0, 1]
).ravel()

vrais_negatifs, faux_positifs, faux_negatifs, vrais_positifs = confusion_matrix(
    y_test, prediction_test, labels=[0, 1]
).ravel()

total_erreurs = (prediction_test != y_test).sum()
print(f"Adélie manqués : {faux_negatifs}")
print(f"Autres manchots classés Adélie : {faux_positifs}")
print(f"Total : {total_erreurs} erreurs sur {len(y_test)} fiches")

assert faux_negatifs + faux_positifs == total_erreurs

Adélie manqués : 0
Autres manchots classés Adélie : 0
Total : 0 erreurs sur 69 fiches


In [25]:
fiche_dream = pd.DataFrame([{
    "bill_length_mm": 39.8,
    "bill_depth_mm": 18.4,
    "flipper_length_mm": 190,
    "body_mass_g": float("nan"),
    "island": "Dream",
    "sex": "female",
}])


classe = retenu.predict(fiche_dream)[0]
print("Adélie" if classe == 1 else "Autre espèce")

Adélie


**Réponse à l’équipe**  
Le modèle classe le manchot de Dream comme Adélie. Il a correctement classé les 69 fiches de test, un résultat encourageant qui laisse une part d’incertitude pour cette nouvelle fiche.